## Step 1 — Setup & Data Download

In [1]:
import os, sys, time, json, gc
import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt
from torch_geometric.data import Data

import config
from data_download import download_raw_data
from graph_builder import build_graph_files, load_graph_files
from model import build_model
from trainer import run_training, evaluate
from results import save_all_results, print_results
from imputation_eval import evaluate_imputation
from fairness import (check_demographic_parity_pre,
                       check_demographic_parity_post,
                       plot_demographic_parity)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device : {device}')
print(f'Dataset: {config.DATASET_NAME}')
print(f'Sources: {config.BATCH_SOURCES}')

# Download data + build train/val/test splits (runs once)
download_raw_data()
print('\n✓ Data ready')


/home/tariq/.local/lib/python3.12/site-packages/torch/__config__.py:9: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._show_config()
/home/tariq/.local/lib/python3.12/site-packages/numba/__init__.py:48: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.3)
  import scipy
/home/tariq/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cpu
Dataset: adult_census
Sources: [('clean', 'none'), ('mcar_10', 'none'), ('mcar_20', 'none'), ('mcar_30', 'none')]
[data] ✓ Clean CSV found : adult_census/adult_census_clean.csv  (3221 KB)
[data] ✓ Train CSV exists : adult_census/adult_census_train.csv  (2255 KB)
[data] ✓ Val   CSV exists : adult_census/adult_census_val.csv  (483 KB)
[data] ✓ Test  CSV exists : adult_census/adult_census_test.csv  (483 KB)

✓ Data ready


## Step 2 — Create All Null Datasets (upfront)

In [3]:
# Collect MCAR rates needed from BATCH_SOURCES
needed_rates = sorted(set(
    int(src.split('_')[1])
    for src, _ in config.BATCH_SOURCES
    if src.startswith('mcar_')
))

if needed_rates:
    # Check which already exist
    rates_to_create = []
    for rate in needed_rates:
        csv_path = config.mcar_train_csv(rate)
        if os.path.exists(csv_path):
            size_kb = os.path.getsize(csv_path) / 1024
            print(f'  ✓ {rate}% MCAR already exists: {csv_path}  ({size_kb:.0f} KB) — skipping')
        else:
            rates_to_create.append(rate)
            print(f'  ○ {rate}% MCAR not found — will create')

    if rates_to_create:
        clean_train_df = pd.read_csv(config.TRAIN_CSV_FILE)
        print(f'\nLoaded training split: {clean_train_df.shape}')

        original_rates = config.MISSING_RATE
        all_rates = sorted(set(config.MISSING_RATE) | set(rates_to_create))
        max_needed = max(rates_to_create)
        config.MISSING_RATE = [r for r in all_rates if r <= max_needed]

        from null_injector import run_null_injection
        run_null_injection(clean_train_df, run_impute=False)

        config.MISSING_RATE = original_rates
    else:
        print('\n✓ All null datasets already exist')
else:
    print('No MCAR sources in BATCH_SOURCES — skipping null injection')

print('\n✓ Null datasets ready')


  ✓ 10% MCAR already exists: adult_census_mcar_10pct/adult_census_mcar_10_train.csv  (2293 KB) — skipping
  ✓ 20% MCAR already exists: adult_census_mcar_20pct/adult_census_mcar_20_train.csv  (2084 KB) — skipping
  ✓ 30% MCAR already exists: adult_census_mcar_30pct/adult_census_mcar_30_train.csv  (1874 KB) — skipping

✓ All null datasets already exist

✓ Null datasets ready


## Step 3 — Run All Sources

Processes each source one by one: graph → train → evaluate → fairness → **cleanup** → next source.

In [4]:
def cleanup(model, data, device, source):
    """Delete model, data, tensors, figures, flush GPU."""
    print(f'\n[cleanup] Cleaning up after {source} ...')
    if model is not None:
        model.cpu()
        del model
    if data is not None:
        data.cpu()
        del data
    plt.close('all')
    n = gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f'[cleanup] ✓ Done ({n} objects freed)\n')


all_summaries = []
t_total = time.time()

for i, (source, imputation) in enumerate(config.BATCH_SOURCES, 1):
    print(f"\n{'█'*70}")
    print(f"  RUN {i}/{len(config.BATCH_SOURCES)}  —  SOURCE='{source}'  IMPUTATION='{imputation}'")
    print(f"{'█'*70}")

    model = None
    data  = None

    try:
        t_start = time.time()

        # ── Reconfigure ────────────────────────────────────────────────────
        config.reconfigure(source, imputation)
        config.set_global_seed()

        # ── Imputation (if needed) ─────────────────────────────────────────
        if config.RESOLVED_IMPUTATION != 'none':
            if not os.path.exists(config.SOURCE_CSV):
                parts = config.SOURCE.split('_')
                rate = int(parts[1])
                raw_csv = config.mcar_csv(rate)
                from imputer import impute
                df_raw = pd.read_csv(raw_csv)
                df_imp = impute(df_raw, strategy=config.RESOLVED_IMPUTATION)
                os.makedirs(config.GRAPH_DIR, exist_ok=True)
                df_imp.to_csv(config.SOURCE_CSV, index=False)
                print(f'✓ Imputed CSV saved → {config.SOURCE_CSV}')
            else:
                print(f'✓ Imputed CSV exists: {config.SOURCE_CSV}')

        # ── Imputation eval ────────────────────────────────────────────────
        evaluate_imputation()

        # ── Build graph ────────────────────────────────────────────────────
        build_graph_files(force=True)

        # ── Load graph ─────────────────────────────────────────────────────
        X, y, edge_index, train_mask, val_mask, test_mask, meta = load_graph_files()

        data = Data(
            x          = torch.tensor(X, dtype=torch.float),
            edge_index = edge_index,
            y          = torch.tensor(y, dtype=torch.long),
            train_mask = train_mask,
            val_mask   = val_mask,
            test_mask  = test_mask,
        ).to(device)

        del X, y, edge_index  # free numpy arrays

        # ── Build model ────────────────────────────────────────────────────
        model = build_model(
            num_features=meta['num_features'],
            num_classes=meta['num_classes'],
        ).to(device)

        # ── Train (force retrain) ──────────────────────────────────────────
        if os.path.exists(config.MODEL_F):
            os.remove(config.MODEL_F)
        history = run_training(model, data, device)

        # ── Evaluate ───────────────────────────────────────────────────────
        eval_results = evaluate(model, data, device)
        print_results(eval_results)
        save_all_results(history, eval_results, meta)

        # ── Fairness ───────────────────────────────────────────────────────
        dp_pre  = check_demographic_parity_pre(test_mask)
        dp_post = check_demographic_parity_post(eval_results)
        plot_demographic_parity(dp_pre, dp_post)

        elapsed = time.time() - t_start

        summary = {
            'source':      source,
            'imputation':  config.RESOLVED_IMPUTATION,
            'test_acc':    round(eval_results['test']['acc'], 4),
            'test_f1':     round(eval_results['test']['f1_macro'], 4),
            'test_auc':    round(eval_results['test']['roc_auc'], 4) if eval_results['test'].get('roc_auc') else None,
            'dp_pre':      dp_pre['dp_difference'],
            'dp_post':     dp_post['dp_difference'],
            'elapsed_sec': round(elapsed, 1),
        }
        all_summaries.append(summary)
        print(f"\n✓ '{source}' done in {elapsed:.1f}s  |  Acc={summary['test_acc']}  F1={summary['test_f1']}  DP={summary['dp_post']}")

    except Exception as exc:
        print(f'\n✗ FAILED: {source} — {exc}')
        import traceback; traceback.print_exc()
        all_summaries.append({'source': source, 'error': str(exc)})

    finally:
        cleanup(model, data, device, source)

print(f"\n{'═'*70}")
print(f"  ALL DONE — {len(config.BATCH_SOURCES)} sources in {time.time()-t_total:.1f}s")
print(f"{'═'*70}")



██████████████████████████████████████████████████████████████████████
  RUN 1/4  —  SOURCE='clean'  IMPUTATION='none'
██████████████████████████████████████████████████████████████████████

[config] ✓ Reconfigured:
[config]   SOURCE           = 'clean'
[config]   IMPUTATION       = 'none'  (No Imputation (raw nulls))
[config]   SOURCE_CSV       = adult_census/adult_census_clean.csv
[config]   GRAPH_DIR        = adult_census/
[config]   FILE_PREFIX      = adult_census_clean
[imputation_eval] Skipped — no imputation to evaluate (SOURCE='clean', imputation='none')
[graph_builder] Building graph files for SOURCE='clean' ...
[graph_builder] Source CSV   : adult_census/adult_census_clean.csv
[graph_builder] Imputation   : none
[graph_builder] Output dir   : adult_census/
[graph_builder] File prefix  : adult_census_clean

[graph_builder] Rows loaded (full clean) : 30,162
[graph_builder] NaN cells                : 0
[graph_builder] Label distribution : <=50K (0)=22,654  >50K (1)=7,508
[graph

/home/tariq/Desktop/FAIR-GNN/gnn_project_v13/gnn_project_v12/graph_builder.py:281: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 0.04279571  0.88028814 -0.03333996 ...  1.48937355 -1.25151078
  1.0325595 ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[obs_mask, col] = scaler.transform(
/home/tariq/Desktop/FAIR-GNN/gnn_project_v13/gnn_project_v12/graph_builder.py:281: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-1.0627216  -1.00787131  0.24469349 ... -0.3585745   0.11070545
  0.92884082]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[obs_mask, col] = scaler.transform(
/home/tariq/Desktop/FAIR-GNN/gnn_project_v13/gnn_project_v12/graph_builder.py:281: FutureWarning: Setting an item of incompatible dtype is deprecated and 

done  →  214,970 edges
[graph_builder] Graph diagnostics :
[graph_builder]   Avg degree : 7.13  Min : 5  Max : 20
[graph_builder]   Graph is fully connected (1 component)
[graph_builder]   ✓ adult_census_clean_node_features.npy  (30162, 14)
[graph_builder]   ✓ adult_census_clean_node_labels.npy    (30162,)
[graph_builder]   ✓ adult_census_clean_edge_index.npy     (2, 214970)
[graph_builder]   ✓ train/val/test mask .npy files
[graph_builder]   ✓ adult_census_clean_null_mask.npy  (30162, 14)  (sentinel cells=0)
[graph_builder]   ✓ adult_census_clean_metadata.json

[graph_builder] ✓ All files saved to: adult_census/
[graph_builder] Loading graph files for SOURCE='clean' ...
[graph_builder]   null_mask     : (30162, 14)  sentinel cells=0
[graph_builder]   node_features : (30162, 14)  dtype=float32  NaNs=0 (should be 0)
[graph_builder]   node_labels   : (30162,)  unique=[0, 1]
[graph_builder]   edge_index    : (2, 214970)
[graph_builder]   train_mask    : 21,125 nodes
[graph_builder]   val_

/home/tariq/Desktop/FAIR-GNN/gnn_project_v13/gnn_project_v12/fairness.py:302: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


[fairness] Saved adult_census_clean_demographic_parity.png

✓ 'clean' done in 41.9s  |  Acc=0.8322  F1=0.7525  DP=0.166613

[cleanup] Cleaning up after clean ...
[cleanup] ✓ Done (20821 objects freed)


██████████████████████████████████████████████████████████████████████
  RUN 2/4  —  SOURCE='mcar_10'  IMPUTATION='none'
██████████████████████████████████████████████████████████████████████

[config] ✓ Reconfigured:
[config]   SOURCE           = 'mcar_10'
[config]   IMPUTATION       = 'none'  (No Imputation (raw nulls))
[config]   SOURCE_CSV       = adult_census_mcar_10pct/adult_census_mcar_10_train.csv
[config]   GRAPH_DIR        = adult_census_mcar_10pct/
[config]   FILE_PREFIX      = adult_census_mcar_10
[imputation_eval] Skipped — no imputation to evaluate (SOURCE='mcar_10', imputation='none')
[graph_builder] Building graph files for SOURCE='mcar_10' ...
[graph_builder] Source CSV   : adult_census_mcar_10pct/adult_census_mcar_10_train.csv
[graph_builder] Imputation   : none
[graph

/home/tariq/Desktop/FAIR-GNN/gnn_project_v13/gnn_project_v12/fairness.py:302: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


[fairness] Saved adult_census_mcar_10_demographic_parity.png

✓ 'mcar_10' done in 136.3s  |  Acc=0.8325  F1=0.7498  DP=0.171701

[cleanup] Cleaning up after mcar_10 ...
[cleanup] ✓ Done (16284 objects freed)


██████████████████████████████████████████████████████████████████████
  RUN 3/4  —  SOURCE='mcar_20'  IMPUTATION='none'
██████████████████████████████████████████████████████████████████████

[config] ✓ Reconfigured:
[config]   SOURCE           = 'mcar_20'
[config]   IMPUTATION       = 'none'  (No Imputation (raw nulls))
[config]   SOURCE_CSV       = adult_census_mcar_20pct/adult_census_mcar_20_train.csv
[config]   GRAPH_DIR        = adult_census_mcar_20pct/
[config]   FILE_PREFIX      = adult_census_mcar_20
[imputation_eval] Skipped — no imputation to evaluate (SOURCE='mcar_20', imputation='none')
[graph_builder] Building graph files for SOURCE='mcar_20' ...
[graph_builder] Source CSV   : adult_census_mcar_20pct/adult_census_mcar_20_train.csv
[graph_builder] Imputation   : none

/home/tariq/Desktop/FAIR-GNN/gnn_project_v13/gnn_project_v12/fairness.py:302: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


[fairness] Saved adult_census_mcar_20_demographic_parity.png

✓ 'mcar_20' done in 142.9s  |  Acc=0.8278  F1=0.7447  DP=0.194007

[cleanup] Cleaning up after mcar_20 ...
[cleanup] ✓ Done (16345 objects freed)


██████████████████████████████████████████████████████████████████████
  RUN 4/4  —  SOURCE='mcar_30'  IMPUTATION='none'
██████████████████████████████████████████████████████████████████████

[config] ✓ Reconfigured:
[config]   SOURCE           = 'mcar_30'
[config]   IMPUTATION       = 'none'  (No Imputation (raw nulls))
[config]   SOURCE_CSV       = adult_census_mcar_30pct/adult_census_mcar_30_train.csv
[config]   GRAPH_DIR        = adult_census_mcar_30pct/
[config]   FILE_PREFIX      = adult_census_mcar_30
[imputation_eval] Skipped — no imputation to evaluate (SOURCE='mcar_30', imputation='none')
[graph_builder] Building graph files for SOURCE='mcar_30' ...
[graph_builder] Source CSV   : adult_census_mcar_30pct/adult_census_mcar_30_train.csv
[graph_builder] Imputation   : none

/home/tariq/Desktop/FAIR-GNN/gnn_project_v13/gnn_project_v12/fairness.py:302: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Comparison Table

In [4]:
print(f"\n{'═'*85}")
print(f"  {'Source':<20} {'Imputation':<12} {'Test Acc':>9} {'Test F1':>9} {'Test AUC':>9} {'DP Pre':>8} {'DP Post':>8} {'Time':>7}")
print(f"  {'─'*80}")

for s in all_summaries:
    if 'error' in s:
        print(f"  {s['source']:<20} FAILED: {s['error']}")
        continue
    print(f"  {s['source']:<20} {s['imputation']:<12} "
          f"{s['test_acc']:>9.4f} {s['test_f1']:>9.4f} "
          f"{s['test_auc']:>9.4f} " if s['test_auc'] else 'N/A',
          f"{s['dp_pre']:>8.4f} {s['dp_post']:>8.4f} {s['elapsed_sec']:>6.0f}s")

print(f"{'═'*85}")

# Save
os.makedirs(config.RESULTS_DIR, exist_ok=True)
batch_json = os.path.join(config.RESULTS_DIR, f'{config.DATASET_NAME}_batch_summary.json')
with open(batch_json, 'w') as f:
    json.dump(all_summaries, f, indent=2)
print(f'\n✓ Saved → {batch_json}')



═════════════════════════════════════════════════════════════════════════════════════
  Source               Imputation    Test Acc   Test F1  Test AUC   DP Pre  DP Post    Time
  ────────────────────────────────────────────────────────────────────────────────
  clean                none            0.8322    0.7525    0.8857    0.1740   0.1666     42s
  mcar_10              knn             0.8309    0.7451    0.8820    0.1740   0.1675     68s
  mcar_20              knn             0.8265    0.7396    0.8831    0.1740   0.1765     77s
  mcar_30              knn             0.8281    0.7380    0.8750    0.1740   0.1958     81s
═════════════════════════════════════════════════════════════════════════════════════

✓ Saved → results/adult_census_batch_summary.json
